# Baseline model
We use ResNet18 to build a simple video classification model

In [15]:
import torch
from torchvision import models
from torch import nn
from pathlib import Path
import csv
import random
import cv2
import math
from PIL import Image, ImageOps
from torchvision import transforms


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Baseline model
We use a pretrained ResNet18 to extract frame features

We keep its weights fixed and train a simple classifier

In [16]:
torch.manual_seed(42)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for parameter in resnet.parameters():
    parameter.requires_grad = False
feature_size = resnet.fc.in_features
resnet.fc = nn.Identity()
classifier = nn.Linear(feature_size, 1)
resnet = resnet.to(device)
classifier = classifier.to(device)
print("Feature size:", feature_size)

Feature size: 512


# Choose videos

We select the same number of positive and negative videos

We use the existing video split for training and validation

In [17]:
videos_per_class = 750  # 5 = 10 videos, 250 = 500, 750 = 1500
sample_fps = 2
dataset_path = Path("../../Nexar_DashCam_Dataset/train")
split_path = Path("video_split.csv")
with split_path.open(encoding="utf-8", newline="") as file:
    split_rows = list(csv.DictReader(file))
train_count = videos_per_class * 4 // 5
validation_count = videos_per_class - train_count
random_generator = random.Random(42)
train_videos = []
validation_videos = []
for label, class_name in [("1", "positive"), ("0", "negative")]:
    for split, count, videos in [("train", train_count, train_videos),
                                 ("validation", validation_count, validation_videos)]:
        ids = sorted(row["video_id"] for row in split_rows
                     if row["label"] == label and row["split"] == split)
        if len(ids) < count:
            raise ValueError(f"Not enough {class_name} videos in {split}")
        random_generator.shuffle(ids)
        videos.extend(dataset_path / class_name / f"{video_id}.mp4" for video_id in ids[:count])
print("Train videos:", len(train_videos))
print("Validation videos:", len(validation_videos))

Train videos: 1200
Validation videos: 300


# Prepare frames
We convert frames to tensors and normalize them for ResNet18

In [ ]:
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

Video: 00921.mp4
Input shape: torch.Size([1, 3, 224, 224])


# Extract video features
We sample frames at a fixed rate and resize them with padding

ResNet18 extracts features from each frame, and we average them for each video

In [21]:
def extract_video_features(video_path):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    source_fps = capture.get(cv2.CAP_PROP_FPS)
    expected_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if not math.isfinite(source_fps) or source_fps <= 0 or sample_fps > source_fps:
        capture.release()
        raise ValueError(f"Invalid video or sampling FPS: {video_path}")
    frame_index = 0
    sampled_count = 0
    batch = []
    frame_features = []
    resnet.eval()
    try:
        with torch.no_grad():
            while True:
                success, frame = capture.read()
                if not success:
                    break
                timestamp = frame_index / source_fps
                if timestamp + 1e-9 >= sampled_count / sample_fps:
                    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                    image = ImageOps.pad(image, (224, 224), color="black")
                    batch.append(preprocess(image))
                    sampled_count += 1
                    if len(batch) == 32:
                        frame_features.append(resnet(torch.stack(batch).to(device)))
                        batch = []
                frame_index += 1
            if batch:
                frame_features.append(resnet(torch.stack(batch).to(device)))
    finally:
        capture.release()
    if not frame_features or (expected_frames > 0 and frame_index < expected_frames):
        raise ValueError(f"Incomplete video: {video_path}")
    return torch.cat(frame_features).mean(dim=0, keepdim=True)

# Prepare data
We extract features and labels for each video

In [24]:
def prepare_data(videos):
    features = []
    labels = []
    for index, video_path in enumerate(videos, start=1):
        features.append(extract_video_features(video_path))
        labels.append(1 if video_path.parent.name == "positive" else 0)
        if index % 10 == 0 or index == len(videos):
            print(f"Processed: {index}/{len(videos)} videos")
    features = torch.cat(features, dim=0)
    labels = torch.tensor(labels, dtype=torch.float32, device=device).unsqueeze(1)
    return features, labels

# Extract training and validation features
We extract features from the selected videos for training and validation

In [25]:
train_features, train_labels = prepare_data(train_videos)
validation_features, validation_labels = prepare_data(validation_videos)
print("Train features:", train_features.shape)
print("Train labels:", train_labels.shape)
print("Validation features:", validation_features.shape)
print("Validation labels:", validation_labels.shape)

Processed: 10/1200 videos
Processed: 20/1200 videos
Processed: 30/1200 videos
Processed: 40/1200 videos
Processed: 50/1200 videos
Processed: 60/1200 videos
Processed: 70/1200 videos
Processed: 80/1200 videos
Processed: 90/1200 videos
Processed: 100/1200 videos
Processed: 110/1200 videos
Processed: 120/1200 videos
Processed: 130/1200 videos
Processed: 140/1200 videos
Processed: 150/1200 videos
Processed: 160/1200 videos
Processed: 170/1200 videos
Processed: 180/1200 videos
Processed: 190/1200 videos
Processed: 200/1200 videos
Processed: 210/1200 videos
Processed: 220/1200 videos
Processed: 230/1200 videos
Processed: 240/1200 videos
Processed: 250/1200 videos
Processed: 260/1200 videos
Processed: 270/1200 videos
Processed: 280/1200 videos
Processed: 290/1200 videos
Processed: 300/1200 videos
Processed: 310/1200 videos
Processed: 320/1200 videos
Processed: 330/1200 videos
Processed: 340/1200 videos
Processed: 350/1200 videos
Processed: 360/1200 videos
Processed: 370/1200 videos
Processed:

# Train the classifier
We train the classifier using the extracted training features

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001)
epochs = 100
classifier.train()
for epoch in range(epochs):
    logits = classifier(train_features)
    loss = criterion(logits, train_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/20 - Loss: 0.6729
Epoch 10/20 - Loss: 0.6581
Epoch 15/20 - Loss: 0.6464
Epoch 20/20 - Loss: 0.6368


# Check validation results
We check the model on validation videos and count its correct and incorrect predictions

In [39]:
classifier.eval()
with torch.no_grad():
    logits = classifier(validation_features)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.45).float()
    accuracy = (predictions == validation_labels).float().mean()
    true_positive = ((predictions == 1) & (validation_labels == 1)).sum()
    true_negative = ((predictions == 0) & (validation_labels == 0)).sum()
    false_positive = ((predictions == 1) & (validation_labels == 0)).sum()
    false_negative = ((predictions == 0) & (validation_labels == 1)).sum()
print("Validation accuracy:", accuracy.item())
print("Positive correctly detected:", true_positive.item())
print("Negative correctly detected:", true_negative.item())
print("False positives:", false_positive.item())
print("False negatives:", false_negative.item())

Validation accuracy: 0.6899999976158142
Positive correctly detected: 108
Negative correctly detected: 99
False positives: 51
False negatives: 42


----
------------
-----------

# Test Dataset(public)
now we measure model accuracy on test dataset

In [41]:
test_path = dataset_path.parent / "test-public"
positive_test = sorted((test_path / "positive").glob("*.mp4"))
negative_test = sorted((test_path / "negative").glob("*.mp4"))
test_videos = positive_test + negative_test
print("Positive test videos:", len(positive_test))
print("Negative test videos:", len(negative_test))
print("Total test videos:", len(test_videos))

Positive test videos: 334
Negative test videos: 333
Total test videos: 667


# Extract test features
We extract test video features using the same method as the training videos

In [42]:
test_features, test_labels = prepare_data(test_videos)
print("Test features:", test_features.shape)
print("Test labels:", test_labels.shape)

Processed: 10/667 videos
Processed: 20/667 videos
Processed: 30/667 videos
Processed: 40/667 videos
Processed: 50/667 videos
Processed: 60/667 videos
Processed: 70/667 videos
Processed: 80/667 videos
Processed: 90/667 videos
Processed: 100/667 videos
Processed: 110/667 videos
Processed: 120/667 videos
Processed: 130/667 videos
Processed: 140/667 videos
Processed: 150/667 videos
Processed: 160/667 videos
Processed: 170/667 videos
Processed: 180/667 videos
Processed: 190/667 videos
Processed: 200/667 videos
Processed: 210/667 videos
Processed: 220/667 videos
Processed: 230/667 videos
Processed: 240/667 videos
Processed: 250/667 videos
Processed: 260/667 videos
Processed: 270/667 videos
Processed: 280/667 videos
Processed: 290/667 videos
Processed: 300/667 videos
Processed: 310/667 videos
Processed: 320/667 videos
Processed: 330/667 videos
Processed: 340/667 videos
Processed: 350/667 videos
Processed: 360/667 videos
Processed: 370/667 videos
Processed: 380/667 videos
Processed: 390/667 vi

# Check test results
We evaluate the trained model on the public test videos

In [43]:
classifier.eval()
with torch.no_grad():
    test_logits = classifier(test_features)
    test_probabilities = torch.sigmoid(test_logits)
    test_predictions = (test_probabilities >= 0.45).float()
    test_accuracy = (test_predictions == test_labels).float().mean()
    true_positive = ((test_predictions == 1) & (test_labels == 1)).sum()
    true_negative = ((test_predictions == 0) & (test_labels == 0)).sum()
    false_positive = ((test_predictions == 1) & (test_labels == 0)).sum()
    false_negative = ((test_predictions == 0) & (test_labels == 1)).sum()
print("Test accuracy:", test_accuracy.item())
print("Positive correctly detected:", true_positive.item())
print("Negative correctly detected:", true_negative.item())
print("False positives:", false_positive.item())
print("False negatives:", false_negative.item())

Test accuracy: 0.6581709384918213
Positive correctly detected: 223
Negative correctly detected: 216
False positives: 117
False negatives: 111
